In [4]:
# Cell 1: Imports and paths
import cv2
import os
import numpy as np
import torch
from pathlib import Path
from deep_sort_realtime.deepsort_tracker import DeepSort
from ultralytics import YOLO
import pandas as pd
from scipy.optimize import linear_sum_assignment

# Base path
basePath = "."
task1ImagesPath = os.path.join(basePath, "Task1/images")
task1GtPath = os.path.join(basePath, "Task1/gt/gt.txt")
task2ImagesPath = os.path.join(basePath, "Task2/images")

In [5]:
# Cell 2: Convert Task1 images to video
frameRate = 14
outputVideoPath = "task1_input.mp4"
images = sorted([img for img in os.listdir(task1ImagesPath) if img.endswith(".jpg")])
sampleImage = cv2.imread(os.path.join(task1ImagesPath, images[0]))
height, width, _ = sampleImage.shape
videoWriter = cv2.VideoWriter(outputVideoPath, cv2.VideoWriter_fourcc(*'mp4v'), frameRate, (width, height))

for imgName in images:
    frame = cv2.imread(os.path.join(task1ImagesPath, imgName))
    videoWriter.write(frame)

videoWriter.release()


In [8]:
# Cell 3: Load YOLOv8 model and DeepSORT tracker
yoloModel = YOLO("yolov8x.pt")  # Change to your trained model if needed
tracker = DeepSort(max_age=30, n_init=3)

In [7]:
# Cell 4: Process video for Task1
cap = cv2.VideoCapture("task1_input.mp4")
outVideoPath = "task1.mp4"
videoWriter = cv2.VideoWriter(outVideoPath, cv2.VideoWriter_fourcc(*'mp4v'), frameRate, (width, height))
trackingResults = []

frameId = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break
    results = yoloModel(frame)[0]
    boxes = results.boxes.xyxy.cpu().numpy()
    confidences = results.boxes.conf.cpu().numpy()
    classes = results.boxes.cls.cpu().numpy()

    detections = []
    for i, cls in enumerate(classes):
        if cls == 0:  # pedestrian class
            x1, y1, x2, y2 = boxes[i]
            detections.append(([x1, y1, x2-x1, y2-y1], confidences[i], "person"))

    tracks = tracker.update_tracks(detections, frame=frame)

    for track in tracks:
        if not track.is_confirmed():
            continue
        trackId = track.track_id
        l, t, w, h = track.to_ltrb()
        l, t, w, h = int(l), int(t), int(w-l), int(h-t)
        trackingResults.append([frameId, trackId, l, t, w, h])
        cv2.rectangle(frame, (l, t), (l+w, t+h), (0,255,0), 2)
        cv2.putText(frame, str(trackId), (l, t-5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 2)

    videoWriter.write(frame)
    frameId += 1

cap.release()
videoWriter.release()

pd.DataFrame(trackingResults, columns=["frame", "id", "bbLeft", "bbTop", "bbWidth", "bbHeight"]).to_csv("task1_tracking.txt", index=False, header=False)


0: 384x640 16 persons, 1 umbrella, 40.2ms
Speed: 1.6ms preprocess, 40.2ms inference, 4.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 18 persons, 1 umbrella, 38.9ms
Speed: 1.4ms preprocess, 38.9ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 16 persons, 1 umbrella, 33.4ms
Speed: 1.4ms preprocess, 33.4ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 15 persons, 1 umbrella, 32.7ms
Speed: 1.4ms preprocess, 32.7ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 14 persons, 1 umbrella, 1 handbag, 54.5ms
Speed: 1.6ms preprocess, 54.5ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 14 persons, 1 umbrella, 1 handbag, 32.2ms
Speed: 1.4ms preprocess, 32.2ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 16 persons, 1 horse, 1 umbrella, 32.3ms
Speed: 1.4ms preprocess, 32.3ms inference, 0.5ms postprocess per image at shape (1, 3,

In [ ]:
# Cell 5: Load ground truth for Task1
gtColumns = ["frame", "id", "bbLeft", "bbTop", "bbWidth", "bbHeight", "conf", "cls", "visibility"]
gtData = pd.read_csv(task1GtPath, sep=",", header=None, names=gtColumns)
gtData = gtData.iloc[:, :6]

In [ ]:
# Cell 6: Compute MOTA
def computeIou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[0]+boxA[2], boxB[0]+boxB[2])
    yB = min(boxA[1]+boxA[3], boxB[1]+boxB[3])
    interArea = max(0, xB-xA) * max(0, yB-yA)
    boxAArea = boxA[2]*boxA[3]
    boxBArea = boxB[2]*boxB[3]
    return interArea / float(boxAArea + boxBArea - interArea)

allFrames = sorted(gtData['frame'].unique())
fp, fn, idsw, gtCount = 0, 0, 0, 0
prevMatches = {}

for f in allFrames:
    gtFrame = gtData[gtData['frame']==f][["id","bbLeft","bbTop","bbWidth","bbHeight"]].values
    predFrame = np.array([r[1:] for r in trackingResults if r[0]==f])

    gtCount += len(gtFrame)

    if len(gtFrame)==0 or len(predFrame)==0:
        fn += len(gtFrame)
        fp += len(predFrame)
        continue

    iouMatrix = np.zeros((len(gtFrame), len(predFrame)))
    for i, gtBox in enumerate(gtFrame):
        for j, predBox in enumerate(predFrame):
            iouMatrix[i,j] = computeIou(gtBox[1:], predBox)

    gtIdx, predIdx = linear_sum_assignment(-iouMatrix)

    matchedGt, matchedPred = set(), set()
    for g, p in zip(gtIdx, predIdx):
        if iouMatrix[g,p] >= 0.5:
            matchedGt.add(g)
            matchedPred.add(p)
            if gtFrame[g,0] in prevMatches and prevMatches[gtFrame[g,0]] != predFrame[p,0]:
                idsw += 1
            prevMatches[gtFrame[g,0]] = predFrame[p,0]

    fn += len(gtFrame) - len(matchedGt)
    fp += len(predFrame) - len(matchedPred)

mota = 1 - (fp + fn + idsw)/gtCount
print(f"MOTA: {mota:.4f}, FP: {fp}, FN: {fn}, IDSW: {idsw}, GT: {gtCount}")

In [ ]:
# Cell 7: Task2 pedestrian counting
images = sorted([img for img in os.listdir(task2ImagesPath) if img.endswith(".jpg")])
tracker = DeepSort(max_age=30, n_init=3)
counts = []

for frameId, imgName in enumerate(images, start=1):
    frame = cv2.imread(os.path.join(task2ImagesPath, imgName))
    results = yoloModel(frame)[0]
    boxes = results.boxes.xyxy.cpu().numpy()
    confidences = results.boxes.conf.cpu().numpy()
    classes = results.boxes.cls.cpu().numpy()

    detections = []
    for i, cls in enumerate(classes):
        if cls == 0:
            x1, y1, x2, y2 = boxes[i]
            detections.append(([x1, y1, x2-x1, y2-y1], confidences[i], "person"))

    tracks = tracker.update_tracks(detections, frame=frame)
    count = sum([1 for t in tracks if t.is_confirmed()])
    counts.append([frameId, count])

    for t in tracks:
        if not t.is_confirmed():
            continue
        l, t_, w, h = t.to_ltrb()
        l, t_, w, h = int(l), int(t_), int(w-l), int(h-t_)
        cv2.rectangle(frame, (l, t_), (l+w, t_+h), (0,255,0), 2)
        cv2.putText(frame, str(t.track_id), (l, t_-5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 2)

    if frameId == 1:
        videoWriter = cv2.VideoWriter("task2.mp4", cv2.VideoWriter_fourcc(*'mp4v'), frameRate, (frame.shape[1], frame.shape[0]))
    videoWriter.write(frame)

videoWriter.release()
pd.DataFrame(counts, columns=["Number","Count"]).to_csv("task2_counts.csv", index=False)